# Descriptive Statistics
## AI Systems Outperform Humans in Fraud Detection and Resistance to Motivated Investor Pressure

This notebook produces all descriptive statistics tables for the Supplementary Materials.

**Contents:**
- Table DS1: AI study — mean warning intensity (Q3) at Turn 1 by model, risk tier, and framing condition
- Table DS2: AI study — mean warning degradation (T1→T2) by model and framing condition (High Risk only)
- Table DS3: AI study — endorsement rates (Q4) and reversal rates by model and risk tier
- Table DS4: Human benchmark — participant demographics
- Table DS5: Human benchmark — investment experience distribution
- Table DS6: Human benchmark — baseline endorsement rates by risk tier and condition
- Table DS7: Human benchmark — warning suppression rates by band and condition (SR and LLM-coded)
- Table DS8: Human benchmark — non-valid response rates by band and condition

**Inputs required:**
- `full_study_results_FINAL.csv` — AI model turn-level results
- `human_benchmark_coded.csv` — human benchmark coded data

**Note:** All tables are exported as CSV files for inclusion in the Supplementary Materials.

In [ ]:
# ============================================================
# CELL 1 — IMPORTS AND SETUP
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import os

matplotlib.rcParams.update({
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

AI_PATH    = 'full_study_results_FINAL.csv'
HUMAN_PATH = 'human_benchmark_coded.csv'

BAND_MAP = {
    'H1': 'Band 1', 'H2': 'Band 1',
    'H3': 'Band 2', 'H5': 'Band 2',
    'H4': 'Band 3', 'H6': 'Band 3',
}
BAND_ORDER  = ['Band 1', 'Band 2', 'Band 3']
MODEL_ORDER = ['claude', 'gpt4o', 'gpt4o_mini', 'gemini', 'deepseek', 'llama', 'grok']
MODEL_LABELS = {
    'claude':     'Claude Sonnet',
    'gpt4o':      'GPT-4o',
    'gpt4o_mini': 'GPT-4o mini',
    'gemini':     'Gemini 2.5 Flash',
    'deepseek':   'DeepSeek V3',
    'llama':      'Llama 3.3 70B',
    'grok':       'Grok 3',
}
TIER_ORDER = ['Low', 'Medium', 'High']

print('✓ Setup complete')

In [ ]:
# ============================================================
# CELL 2 — LOAD DATA
# ============================================================

df_ai = pd.read_csv(AI_PATH)
df_ai['model_label'] = df_ai['model'].map(MODEL_LABELS)
df_ai['risk_tier']   = pd.Categorical(df_ai['risk_tier'], categories=TIER_ORDER, ordered=True)

df_h = pd.read_csv(HUMAN_PATH)

# Recode demographic columns (strip non-breaking spaces from column names)
df_h.columns = [c.strip().replace('\xa0', '') for c in df_h.columns]

# Financial literacy scoring (corrected: Qualtrics exports TRUE/FALSE in caps)
CORRECT_ANSWERS = {
    'FL_Q1': 'More than $102',
    'FL_Q2': 'Less than today',
    'FL_Q3': 'They will fall',
    'FL_Q4': 'TRUE',
    'FL_Q5': 'FALSE',
}
fl_item_cols = []
for col, correct in CORRECT_ANSWERS.items():
    # Handle trailing spaces in column names
    matching = [c for c in df_h.columns if c.strip() == col]
    if matching:
        actual_col = matching[0]
        item_name  = f'fl_{col.lower()}_correct'
        df_h[item_name] = (df_h[actual_col].str.strip() == correct).astype(float)
        df_h.loc[df_h[actual_col].isna(), item_name] = np.nan
        fl_item_cols.append(item_name)
    else:
        print(f'  ⚠ Column {col!r} not found')

df_h['literacy_score'] = df_h[fl_item_cols].sum(axis=1, min_count=1)
df_h['high_literacy']  = (df_h['literacy_score'] >= 4).astype(float)

# Band mapping for human data
df_h['band'] = df_h['scenario_id'].map(BAND_MAP)

# Self-reported Q2 suppression
def parse_q2(series):
    def _p(v):
        if pd.isna(v): return np.nan
        s = str(v).strip().lower()
        if s in ('1', 'yes', 'true'):  return 1
        if s in ('0', 'no', 'false'): return 0
        return np.nan
    return series.apply(_p)

df_h['sr_t1_Q2'] = parse_q2(df_h['human_t1_Q2'])
df_h['sr_t2_Q2'] = parse_q2(df_h['human_t2_Q2'])
df_h['suppressed_sr'] = (
    (df_h['sr_t1_Q2'] == 1) & (df_h['sr_t2_Q2'] == 0)
).astype(float)
df_h.loc[df_h['sr_t2_Q2'].isna(), 'suppressed_sr'] = np.nan

print(f'AI data:    {df_ai.shape[0]:,} rows')
print(f'Human data: {df_h.shape[0]:,} rows')
print(f'Literacy score range: {df_h["literacy_score"].min():.0f}–{df_h["literacy_score"].max():.0f}')

In [ ]:
# ============================================================
# CELL 3 — TABLE DS1
# AI study: mean Q3 (warning intensity) at Turn 1
# by model, risk tier, and framing condition
# ============================================================

print('=' * 65)
print('TABLE DS1: Mean Warning Intensity (Q3) at Turn 1')
print('By model, risk tier, and framing condition')
print('=' * 65)

t1 = df_ai[df_ai['turn'] == 1].copy()

ds1_rows = []
for model in MODEL_ORDER:
    label = MODEL_LABELS[model]
    for tier in TIER_ORDER:
        for cond in ['neutral', 'motivated']:
            sub = t1[
                (t1['model'] == model) &
                (t1['risk_tier'] == tier) &
                (t1['t1_condition'] == cond) &
                t1['error'].isna()
            ]['Q3'].dropna()
            if len(sub) == 0:
                continue
            ci = 1.96 * sub.sem()
            ds1_rows.append({
                'Model':       label,
                'Risk tier':   tier,
                'Condition':   cond.capitalize(),
                'N':           len(sub),
                'Mean Q3':     round(sub.mean(), 3),
                'SD':          round(sub.std(), 3),
                '95% CI lower': round(sub.mean() - ci, 3),
                '95% CI upper': round(sub.mean() + ci, 3),
            })

ds1 = pd.DataFrame(ds1_rows)
print(ds1.to_string(index=False))
ds1.to_csv('table_ds1_ai_warning_intensity_t1.csv', index=False)
print('\nSaved: table_ds1_ai_warning_intensity_t1.csv')

In [ ]:
# ============================================================
# CELL 4 — TABLE DS2
# AI study: mean warning degradation (T1→T2)
# by model and framing condition, High Risk only
# ============================================================

print('=' * 65)
print('TABLE DS2: Mean Warning Degradation (Q3 T1 − Q3 T2)')
print('High Risk only, by model and framing condition')
print('Positive = weakening; negative = strengthening')
print('=' * 65)

# Pivot to get T1 and T2 Q3 per run
hr = df_ai[
    (df_ai['risk_tier'] == 'High') &
    df_ai['error'].isna()
].copy()

hr_wide = hr.pivot_table(
    index=['run_id', 'model', 't1_condition'],
    columns='turn',
    values='Q3',
    aggfunc='first'
).reset_index()
hr_wide.columns = ['run_id', 'model', 't1_condition', 'Q3_t1', 'Q3_t2', 'Q3_t3']
hr_wide['degradation_t2'] = hr_wide['Q3_t1'] - hr_wide['Q3_t2']
hr_wide['degradation_t3'] = hr_wide['Q3_t1'] - hr_wide['Q3_t3']

ds2_rows = []
for model in MODEL_ORDER:
    label = MODEL_LABELS[model]
    for cond in ['neutral', 'motivated']:
        sub = hr_wide[
            (hr_wide['model'] == model) &
            (hr_wide['t1_condition'] == cond)
        ]['degradation_t2'].dropna()
        if len(sub) == 0:
            continue
        ci = 1.96 * sub.sem()
        ds2_rows.append({
            'Model':        label,
            'Condition':    cond.capitalize(),
            'N':            len(sub),
            'Mean degradation': round(sub.mean(), 3),
            'SD':           round(sub.std(), 3),
            '95% CI lower': round(sub.mean() - ci, 3),
            '95% CI upper': round(sub.mean() + ci, 3),
        })

ds2 = pd.DataFrame(ds2_rows)
print(ds2.to_string(index=False))
ds2.to_csv('table_ds2_ai_warning_degradation.csv', index=False)
print('\nSaved: table_ds2_ai_warning_degradation.csv')

In [ ]:
# ============================================================
# CELL 5 — TABLE DS3
# AI study: endorsement rates (Q4) and reversal rates
# by model and risk tier
# ============================================================

print('=' * 65)
print('TABLE DS3: Endorsement Rates (Q4) and Reversal Rates')
print('By model and risk tier')
print('=' * 65)

t1_ai = df_ai[(df_ai['turn'] == 1) & df_ai['error'].isna()].copy()

ds3_rows = []
for model in MODEL_ORDER:
    label = MODEL_LABELS[model]
    for tier in TIER_ORDER:
        sub = t1_ai[
            (t1_ai['model'] == model) &
            (t1_ai['risk_tier'] == tier)
        ]
        if len(sub) == 0:
            continue
        # Turn 2+3 reversal for High Risk
        rev_sub = df_ai[
            (df_ai['model'] == model) &
            (df_ai['risk_tier'] == tier) &
            (df_ai['turn'].isin([2, 3])) &
            df_ai['error'].isna()
        ]
        rev_rate = rev_sub['endorsement_reversal'].mean() if len(rev_sub) > 0 else np.nan

        ds3_rows.append({
            'Model':              label,
            'Risk tier':          tier,
            'N (T1)':             len(sub),
            'T1 endorsement rate': round(sub['Q4'].mean(), 3),
            'T1 warning rate':    round(sub['Q2'].mean(), 3),
            'Reversal rate (T2+T3)': round(rev_rate, 3) if pd.notna(rev_rate) else '—',
        })

ds3 = pd.DataFrame(ds3_rows)
print(ds3.to_string(index=False))
ds3.to_csv('table_ds3_ai_endorsement_reversal.csv', index=False)
print('\nSaved: table_ds3_ai_endorsement_reversal.csv')

In [ ]:
# ============================================================
# CELL 6 — TABLE DS4
# Human benchmark: participant demographics
# ============================================================

print('=' * 65)
print('TABLE DS4: Participant Demographics (N=1,201)')
print('Human benchmark, Prolific US adult sample, 21 April 2026')
print('=' * 65)

N = len(df_h)

# Age
age_col = [c for c in df_h.columns if c.strip() == 'Age'][0]
age     = pd.to_numeric(df_h[age_col], errors='coerce')
print(f'\nAge (N={age.notna().sum()}): Mean={age.mean():.1f} SD={age.std():.1f} Median={age.median():.0f} Range={age.min():.0f}–{age.max():.0f}')

# Gender
gen_col = [c for c in df_h.columns if c.strip() == 'Gender'][0]
gender  = df_h[gen_col].value_counts(dropna=False)
print('\nGender:')
for label, n in gender.items():
    print(f'  {label:<35} {n:>5}  ({100*n/N:.1f}%)')

# Education
edu_col = [c for c in df_h.columns if c.strip() == 'Educ'][0]
educ_order = [
    'Some high school',
    'High school graduate or GED equivalent',
    'Some undergraduate',
    'Completed undergraduate',
    'Some graduate',
    "Completed graduate (Master's or PhD)",
    'Other',
    'Prefer not to say',
]
educ = df_h[edu_col].value_counts(dropna=False).reindex(educ_order, fill_value=0)
print('\nEducation:')
for label, n in educ.items():
    print(f'  {label:<45} {n:>5}  ({100*n/N:.1f}%)')

# Financial literacy
print('\nFinancial literacy score (0–5):')
lit_dist = df_h['literacy_score'].value_counts(dropna=False).sort_index()
for score, n in lit_dist.items():
    print(f'  Score {score:.0f}:  {n:>5}  ({100*n/N:.1f}%)')
n_hilit  = int(df_h['high_literacy'].eq(1).sum())
pct_hilit = 100 * df_h['high_literacy'].mean()
print(f'  High-literacy (>=4): n={n_hilit} ({pct_hilit:.1f}%)')

# Build summary table using pre-computed variables (no backslashes in f-strings)
n_male    = int((df_h[gen_col] == 'Male').sum())
n_female  = int((df_h[gen_col] == 'Female').sum())
n_other   = int((~df_h[gen_col].isin(['Male', 'Female'])).sum())
n_hs      = educ.get('Some high school', 0) + educ.get('High school graduate or GED equivalent', 0)
n_ug      = educ.get('Some undergraduate', 0) + educ.get('Completed undergraduate', 0)
grad_key  = "Completed graduate (Master's or PhD)"
n_grad    = educ.get('Some graduate', 0) + educ.get(grad_key, 0)
lit_mean  = df_h['literacy_score'].mean()
lit_sd    = df_h['literacy_score'].std()

ds4_rows = [
    {'Variable': 'Age — Mean (SD)',                          'Value': f'{age.mean():.1f} ({age.std():.1f})'},
    {'Variable': 'Age — Median',                             'Value': f'{age.median():.0f}'},
    {'Variable': 'Age — Range',                              'Value': f'{int(age.min())}–{int(age.max())}'},
    {'Variable': 'Gender — Male',                            'Value': f'{n_male} ({100*n_male/N:.1f}%)'},
    {'Variable': 'Gender — Female',                          'Value': f'{n_female} ({100*n_female/N:.1f}%)'},
    {'Variable': 'Gender — Non-binary/other',                'Value': f'{n_other} ({100*n_other/N:.1f}%)'},
    {'Variable': 'Education — High school or below',         'Value': f'{n_hs} ({100*n_hs/N:.1f}%)'},
    {'Variable': 'Education — Some/completed undergraduate', 'Value': f'{n_ug} ({100*n_ug/N:.1f}%)'},
    {'Variable': 'Education — Some/completed graduate',      'Value': f'{n_grad} ({100*n_grad/N:.1f}%)'},
    {'Variable': 'Financial literacy — Mean (SD)',           'Value': f'{lit_mean:.2f} ({lit_sd:.2f})'},
    {'Variable': 'Financial literacy — High (>=4)',          'Value': f'{n_hilit} ({pct_hilit:.1f}%)'},
]
ds4 = pd.DataFrame(ds4_rows)
ds4.to_csv('table_ds4_demographics.csv', index=False)
print('\nSaved: table_ds4_demographics.csv')



In [ ]:
# ============================================================
# CELL 7 — TABLE DS5
# Human benchmark: investment experience distribution
# ============================================================

print('=' * 65)
print('TABLE DS5: Investment Experience Distribution (N=1,201)')
print('=' * 65)

inv_col = [c for c in df_h.columns if 'investment_exp' in c.lower()][0]
exp_order = [
    'No experience — I have never invested',
    'Beginner — I have some basic knowledge but limited experience',
    'Intermediate — I invest occasionally and understand the basics',
    'Experienced — I invest regularly and am comfortable with financial products',
    'Expert — I have professional or extensive personal investment experience',
]
inv_exp = df_h[inv_col].value_counts(dropna=False)

ds5_rows = []
for label in exp_order:
    n = inv_exp.get(label, 0)
    # Shorten label for display
    short = label.split(' — ')[0]
    print(f'  {short:<15} {n:>5}  ({100*n/N:.1f}%)')
    ds5_rows.append({'Category': label, 'N': n, '%': round(100*n/N, 1)})

ds5 = pd.DataFrame(ds5_rows)
ds5.to_csv('table_ds5_investment_experience.csv', index=False)
print('\nSaved: table_ds5_investment_experience.csv')

In [ ]:
# ============================================================
# CELL 8 — TABLE DS6
# Human benchmark: baseline endorsement rates
# by risk tier and Turn 1 condition
# ============================================================

print('=' * 65)
print('TABLE DS6: Human Baseline Endorsement Rates')
print('By risk tier and Turn 1 condition')
print('Endorsement = Q4=1 at Turn 1 (investment recommended)')
print('=' * 65)

# Parse T1 Q4
def parse_q4(series):
    def _p(v):
        if pd.isna(v): return np.nan
        s = str(v).strip().lower()
        if s in ('1', 'yes', 'true'):  return 1
        if s in ('0', 'no', 'false'): return 0
        # Handle "1 — ..." style labels
        if s and s[0] == '1': return 1
        if s and s[0] == '0': return 0
        return np.nan
    return series.apply(_p)

df_h['sr_t1_Q4'] = parse_q4(df_h['human_t1_Q4'])

ds6_rows = []
for tier in TIER_ORDER:
    for cond in ['neutral', 'motivated']:
        sub = df_h[
            (df_h['risk_tier'] == tier) &
            (df_h['t1_condition'] == cond) &
            df_h['sr_t1_Q4'].notna()
        ]
        if len(sub) == 0:
            continue
        rate = sub['sr_t1_Q4'].mean()
        ci   = 1.96 * np.sqrt(rate * (1 - rate) / len(sub))
        ds6_rows.append({
            'Risk tier':      tier,
            'Condition':      cond.capitalize(),
            'N':              len(sub),
            'Endorsement rate': round(rate, 3),
            '95% CI lower':   round(rate - ci, 3),
            '95% CI upper':   round(rate + ci, 3),
        })

ds6 = pd.DataFrame(ds6_rows)
print(ds6.to_string(index=False))
ds6.to_csv('table_ds6_human_endorsement_baseline.csv', index=False)
print('\nSaved: table_ds6_human_endorsement_baseline.csv')

In [ ]:
# ============================================================
# CELL 9 — TABLE DS7
# Human benchmark: warning suppression rates
# by band and condition (SR and LLM-coded)
# ============================================================

print('=' * 65)
print('TABLE DS7: Human Warning Suppression Rates')
print('High Risk only, by band and condition')
print('SR = self-reported Q2 (primary); LLM = judge-coded Q2 (cross-val)')
print('=' * 65)

hr_h = df_h[
    (df_h['risk_tier'] == 'High') &
    (df_h['path'] == 'suppression_test')
].copy()

# LLM-coded suppression
hr_h['suppressed_llm'] = (
    (hr_h['llm_t1_Q2'] == 1) & (hr_h['llm_t2_Q2'] == 0)
).astype(float)
hr_h.loc[
    (hr_h['response_quality_t2'] != 'valid') | hr_h['llm_t2_Q2'].isna(),
    'suppressed_llm'
] = np.nan

ds7_rows = []
for band in BAND_ORDER:
    for cond in ['neutral', 'motivated']:
        sub = hr_h[
            (hr_h['band'] == band) &
            (hr_h['t1_condition'] == cond)
        ]
        # SR
        sr_sub  = sub[sub['suppressed_sr'].notna()]
        sr_rate = sr_sub['suppressed_sr'].mean()
        sr_ci   = 1.96 * np.sqrt(sr_rate * (1 - sr_rate) / len(sr_sub)) if len(sr_sub) > 0 else np.nan
        # LLM
        llm_sub  = sub[sub['suppressed_llm'].notna()]
        llm_rate = llm_sub['suppressed_llm'].mean() if len(llm_sub) > 0 else np.nan
        llm_ci   = 1.96 * np.sqrt(llm_rate * (1 - llm_rate) / len(llm_sub)) if len(llm_sub) > 0 else np.nan

        ds7_rows.append({
            'Band':              band,
            'Condition':         cond.capitalize(),
            'N (SR)':            len(sr_sub),
            'SR suppression rate': round(sr_rate, 3) if pd.notna(sr_rate) else np.nan,
            'SR 95% CI lower':   round(sr_rate - sr_ci, 3) if pd.notna(sr_ci) else np.nan,
            'SR 95% CI upper':   round(sr_rate + sr_ci, 3) if pd.notna(sr_ci) else np.nan,
            'N (LLM)':           len(llm_sub),
            'LLM suppression rate': round(llm_rate, 3) if pd.notna(llm_rate) else np.nan,
            'LLM 95% CI lower':  round(llm_rate - llm_ci, 3) if pd.notna(llm_ci) else np.nan,
            'LLM 95% CI upper':  round(llm_rate + llm_ci, 3) if pd.notna(llm_ci) else np.nan,
        })

ds7 = pd.DataFrame(ds7_rows)
print(ds7.to_string(index=False))
ds7.to_csv('table_ds7_human_suppression_rates.csv', index=False)
print('\nSaved: table_ds7_human_suppression_rates.csv')

In [ ]:
# ============================================================
# CELL 10 — TABLE DS8
# Human benchmark: non-valid response rates
# by band and condition
# ============================================================

print('=' * 65)
print('TABLE DS8: Non-valid Turn 2 Response Rates')
print('High Risk, suppression-test path only, by band and condition')
print('Non-valid = off_topic / too_short / gibberish / no_assessment')
print('=' * 65)

hr_h['nonvalid_t2'] = (
    ~hr_h['response_quality_t2'].isin(['valid'])
).astype(float)
hr_h.loc[hr_h['response_quality_t2'].isna(), 'nonvalid_t2'] = np.nan

ds8_rows = []
for band in BAND_ORDER:
    for cond in ['neutral', 'motivated']:
        sub = hr_h[
            (hr_h['band'] == band) &
            (hr_h['t1_condition'] == cond) &
            hr_h['nonvalid_t2'].notna()
        ]
        if len(sub) == 0:
            continue
        rate = sub['nonvalid_t2'].mean()
        ci   = 1.96 * np.sqrt(rate * (1 - rate) / len(sub))

        # Breakdown by quality label
        qual_counts = sub['response_quality_t2'].value_counts()

        ds8_rows.append({
            'Band':              band,
            'Condition':         cond.capitalize(),
            'N':                 len(sub),
            'Non-valid rate':    round(rate, 3),
            '95% CI lower':      round(rate - ci, 3),
            '95% CI upper':      round(rate + ci, 3),
            'n valid':           qual_counts.get('valid', 0),
            'n off_topic':       qual_counts.get('off_topic', 0),
            'n too_short':       qual_counts.get('too_short', 0),
            'n no_assessment':   qual_counts.get('no_assessment', 0),
            'n gibberish':       qual_counts.get('gibberish', 0),
        })

ds8 = pd.DataFrame(ds8_rows)
print(ds8.to_string(index=False))
ds8.to_csv('table_ds8_nonvalid_responses.csv', index=False)
print('\nSaved: table_ds8_nonvalid_responses.csv')

In [ ]:
# ============================================================
# CELL 11 — SUMMARY
# ============================================================

print('=' * 65)
print('ALL DESCRIPTIVE STATISTICS TABLES SAVED')
print('=' * 65)
files = [
    ('table_ds1_ai_warning_intensity_t1.csv',    'DS1: AI warning intensity at Turn 1'),
    ('table_ds2_ai_warning_degradation.csv',     'DS2: AI warning degradation T1→T2'),
    ('table_ds3_ai_endorsement_reversal.csv',    'DS3: AI endorsement and reversal rates'),
    ('table_ds4_demographics.csv',               'DS4: Human benchmark demographics'),
    ('table_ds5_investment_experience.csv',      'DS5: Investment experience distribution'),
    ('table_ds6_human_endorsement_baseline.csv', 'DS6: Human baseline endorsement rates'),
    ('table_ds7_human_suppression_rates.csv',    'DS7: Human warning suppression rates'),
    ('table_ds8_nonvalid_responses.csv',         'DS8: Non-valid Turn 2 response rates'),
]
for fname, label in files:
    exists = '✓' if os.path.exists(fname) else '✗ MISSING'
    print(f'  {exists}  {label}')
    print(f'       {fname}')